# v11 Settlement — Four Expanding Folds

This notebook compares a more robust four-fold time-series validation scheme against the original two-fold v11 settlement model. The 2026 test set remains untouched.

| Fold | Training years | Validation year | Weight |
| --- | --- | --- | ---: |
| 1 | 2021 | 2022 | 25% |
| 2 | 2021–2022 | 2023 | 25% |
| 3 | 2021–2023 | 2024 | 25% |
| 4 | 2021–2024 | 2025 | 25% |

In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise RuntimeError('Could not locate the weather-research repository root.')


REPO_ROOT = find_repo_root(Path.cwd())
RUNNER = REPO_ROOT / 'scripts/run_station_stacking_v11_settlement_expanding_4fold.py'
OUTPUT_DIR = REPO_ROOT / 'data/calibration/station_stacking_v11_settlement_expanding_4fold'
ORIGINAL_DIR = REPO_ROOT / 'data/calibration/station_stacking_v11_settlement'
REPO_ROOT

WindowsPath('D:/dev/weather-research')

## Configuration

Full optimization uses 30 trials per base learner and 30 ridge-stack trials. Use fast mode only for a pipeline check, not for the final comparison.

In [2]:
STATIONS = ('KATL', 'KDAL')
OPTUNA_TRIALS = 30
STARTUP_TRIALS = 15
STACK_OPTUNA_TRIALS = 30
STACK_STARTUP_TRIALS = 15
FAST_MODE = False
EXPORT_MODELS = True

CATBOOST_MAX_ITERATIONS = 1200
CATBOOST_MAX_DEPTH = 8
CATBOOST_MIN_LEARNING_RATE = 0.005
CATBOOST_MAX_BORDER_COUNT = 128

FOLDS = pd.DataFrame(
    [
        {'fold': 1, 'train': '2021', 'validate': 2022, 'weight': 0.25},
        {'fold': 2, 'train': '2021–2022', 'validate': 2023, 'weight': 0.25},
        {'fold': 3, 'train': '2021–2023', 'validate': 2024, 'weight': 0.25},
        {'fold': 4, 'train': '2021–2024', 'validate': 2025, 'weight': 0.25},
    ]
)
display(FOLDS)

,fold,train,validate,weight
0,1,2021,2022,0.25
1,2,2021–2022,2023,0.25
2,3,2021–2023,2024,0.25
3,4,2021–2024,2025,0.25


## Run training

Do not run this cell while another station-stacking experiment is active against the same machine unless you intentionally want concurrent CPU load. The command resumes from Optuna SQLite checkpoints if interrupted.

In [3]:
command = [
    sys.executable,
    str(RUNNER),
    '--stations', ','.join(STATIONS),
    '--optuna-trials', str(OPTUNA_TRIALS),
    '--startup-trials', str(STARTUP_TRIALS),
    '--stack-optuna-trials', str(STACK_OPTUNA_TRIALS),
    '--stack-startup-trials', str(STACK_STARTUP_TRIALS),
    '--catboost-max-iterations', str(CATBOOST_MAX_ITERATIONS),
    '--catboost-max-depth', str(CATBOOST_MAX_DEPTH),
    '--catboost-min-learning-rate', str(CATBOOST_MIN_LEARNING_RATE),
    '--catboost-max-border-count', str(CATBOOST_MAX_BORDER_COUNT),
    '--output-dir', str(OUTPUT_DIR),
    '--quiet-optuna',
]
if FAST_MODE:
    command.append('--fast-mode')
if not EXPORT_MODELS:
    command.append('--skip-export')

print(' '.join(command))
subprocess.run(command, cwd=REPO_ROOT, check=True)

d:\dev\weather-research\.venv\Scripts\python.exe D:\dev\weather-research\scripts\run_station_stacking_v11_settlement_expanding_4fold.py --stations KATL,KDAL --optuna-trials 30 --startup-trials 15 --stack-optuna-trials 30 --stack-startup-trials 15 --catboost-max-iterations 1200 --catboost-max-depth 8 --catboost-min-learning-rate 0.005 --catboost-max-border-count 128 --output-dir D:\dev\weather-research\data\calibration\station_stacking_v11_settlement_expanding_4fold --quiet-optuna


CompletedProcess(args=['d:\\dev\\weather-research\\.venv\\Scripts\\python.exe', 'D:\\dev\\weather-research\\scripts\\run_station_stacking_v11_settlement_expanding_4fold.py', '--stations', 'KATL,KDAL', '--optuna-trials', '30', '--startup-trials', '15', '--stack-optuna-trials', '30', '--stack-startup-trials', '15', '--catboost-max-iterations', '1200', '--catboost-max-depth', '8', '--catboost-min-learning-rate', '0.005', '--catboost-max-border-count', '128', '--output-dir', 'D:\\dev\\weather-research\\data\\calibration\\station_stacking_v11_settlement_expanding_4fold', '--quiet-optuna'], returncode=0)

## Per-fold validation diagnostics

In [4]:
fold_rows = []
for station in STATIONS:
    path = OUTPUT_DIR / f'{station}_year_split_validation_predictions.csv'
    predictions = pd.read_csv(path)
    predictions['absolute_error_f'] = (predictions['actual_high_f'] - predictions['predicted_high_f']).abs()
    summary = (
        predictions.groupby(['fold', 'method'], as_index=False)
        .agg(count=('absolute_error_f', 'size'), mae_f=('absolute_error_f', 'mean'))
    )
    summary['station_id'] = station
    fold_rows.append(summary)

fold_diagnostics = pd.concat(fold_rows, ignore_index=True)
display(
    fold_diagnostics.loc[fold_diagnostics['method'].isin(['xgboost', 'lightgbm', 'catboost'])]
    .sort_values(['station_id', 'fold', 'mae_f'])
)

,fold,method,count,mae_f,station_id
7,fold_2021_2022_to_2023,xgboost,364,1.444933,KATL
0,fold_2021_2022_to_2023,catboost,364,1.500950,KATL
3,fold_2021_2022_to_2023,lightgbm,364,1.517394,KATL
11,fold_2021_2023_to_2024,lightgbm,366,1.234985,KATL
8,fold_2021_2023_to_2024,catboost,366,1.285336,KATL
15,fold_2021_2023_to_2024,xgboost,366,1.287046,KATL
19,fold_2021_2024_to_2025,lightgbm,365,1.333751,KATL
23,fold_2021_2024_to_2025,xgboost,365,1.347310,KATL
16,fold_2021_2024_to_2025,catboost,365,1.375402,KATL
27,fold_2021_to_2022,lightgbm,365,1.443504,KATL


## Untouched 2026 test results

In [5]:
test_rows = []
for station in STATIONS:
    scoreboard = pd.read_csv(OUTPUT_DIR / f'{station}_year_split_scoreboard.csv')
    selected = scoreboard.loc[scoreboard['period'].eq('test_2026')].copy()
    selected['station_id'] = station
    test_rows.append(selected)

test_results = pd.concat(test_rows, ignore_index=True)
display(
    test_results.sort_values(['station_id', 'mae_f'])
    .style.format({'mae_f': '{:.3f}', 'rmse_f': '{:.3f}'})
)

AttributeError: The '.style' accessor requires jinja2

## Four-fold versus original two-fold ridge stack

In [6]:
comparison_rows = []
for station in STATIONS:
    for scheme, root in [('original_2fold', ORIGINAL_DIR), ('expanding_4fold', OUTPUT_DIR)]:
        scoreboard = pd.read_csv(root / f'{station}_year_split_scoreboard.csv')
        bracket = pd.read_csv(root / f'{station}_year_split_bracket_metrics.csv')
        ridge = scoreboard.loc[
            scoreboard['period'].eq('test_2026') & scoreboard['method'].eq('ridge_stack')
        ].iloc[0]
        ridge_bracket = bracket.loc[bracket['method'].eq('ridge_stack')].iloc[0]
        comparison_rows.append(
            {
                'station_id': station,
                'scheme': scheme,
                'count': int(ridge['count']),
                'mae_f': float(ridge['mae_f']),
                'rmse_f': float(ridge['rmse_f']),
                'bucket_hit_pct': float(ridge_bracket['bracket_accuracy_pct']),
            }
        )

comparison = pd.DataFrame(comparison_rows).sort_values(['station_id', 'mae_f'])
display(
    comparison.style.format(
        {'mae_f': '{:.3f}', 'rmse_f': '{:.3f}', 'bucket_hit_pct': '{:.2f}%'}
    )
)

AttributeError: The '.style' accessor requires jinja2

In [7]:
wide = comparison.pivot(index='station_id', columns='scheme', values=['mae_f', 'rmse_f', 'bucket_hit_pct'])
deltas = pd.DataFrame(index=wide.index)
deltas['mae_change_f'] = wide[('mae_f', 'expanding_4fold')] - wide[('mae_f', 'original_2fold')]
deltas['rmse_change_f'] = wide[('rmse_f', 'expanding_4fold')] - wide[('rmse_f', 'original_2fold')]
deltas['bucket_hit_change_pp'] = (
    wide[('bucket_hit_pct', 'expanding_4fold')] - wide[('bucket_hit_pct', 'original_2fold')]
)
display(deltas.style.format('{:+.3f}'))

AttributeError: The '.style' accessor requires jinja2

## Exported artifact checks

In [8]:
artifact_rows = []
for station in STATIONS:
    model_dir = OUTPUT_DIR / 'model_weights'
    bundles = list(model_dir.glob(f'{station}_*.joblib'))
    manifests = list(model_dir.glob(f'{station}_*.json'))
    artifact_rows.append(
        {
            'station_id': station,
            'bundle_ok': len(bundles) == 1 if EXPORT_MODELS else None,
            'manifest_ok': len(manifests) == 1 if EXPORT_MODELS else None,
            'bundle': str(bundles[0]) if bundles else None,
        }
    )
display(pd.DataFrame(artifact_rows))

,station_id,bundle_ok,manifest_ok,bundle
0,KATL,True,True,D:\dev\weather-research\data\calibration\stati...
1,KDAL,True,True,D:\dev\weather-research\data\calibration\stati...
